# LSTM exploration and visualization

Earlier PyTorch experiment that generated the archived forecast figure. This is a separate artifact from the final submission.

Model logic is preserved from the course project. Paths are relative to this repository; generated files go to `outputs/lstm-exploration/`. Saved outputs were cleared for publication.


In [ ]:
from pathlib import Path

# Locate the repository when launched from its root or notebooks directory.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from the repository root.")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lstm-exploration"
for name in ("train.csv", "val.csv", "test.csv"):
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(f"Missing {DATA_DIR / name}. See data/README.md.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 深度学习库
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)

# ============================================================================
# 1. 数据加载和预处理
# ============================================================================

def load_data(file_path):
    """加载数据并添加周序列标签"""
    df = pd.read_csv(file_path, sep='\t')  # 尝试tab分隔
    if len(df.columns) == 1:
        df = pd.read_csv(file_path)  # 如果tab不行，用逗号
    
    # 添加周序列索引
    df['Week'] = range(len(df))
    
    # 重命名列以确保一致性
    if 'Predict Target' in df.columns:
        df.rename(columns={'Predict Target': 'Target'}, inplace=True)
    
    return df

# ============================================================================
# 2. 特征工程
# ============================================================================

def create_features(df):
    """创建时间序列特征"""
    df = df.copy()
    
    # 滞后特征
    for lag in [1, 2, 3, 4, 5, 6, 7, 8, 12, 16, 20, 24]:
        df[f'lag_{lag}'] = df['Target'].shift(lag)
    
    # 滚动统计特征
    for window in [3, 5, 7, 14, 21]:
        df[f'rolling_mean_{window}'] = df['Target'].shift(1).rolling(window=window).mean()
        df[f'rolling_std_{window}'] = df['Target'].shift(1).rolling(window=window).std()
        df[f'rolling_min_{window}'] = df['Target'].shift(1).rolling(window=window).min()
        df[f'rolling_max_{window}'] = df['Target'].shift(1).rolling(window=window).max()
    
    # 差分特征
    df['diff_1'] = df['Target'].diff(1)
    df['diff_2'] = df['Target'].diff(2)
    
    # 周期性特征（假设年周期为52周）
    df['week_of_year'] = df['Week'] % 52
    df['sin_week'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
    df['cos_week'] = np.cos(2 * np.pi * df['week_of_year'] / 52)
    
    return df

# ============================================================================
# 3. 数据集类定义
# ============================================================================

class TimeSeriesDataset(Dataset):
    """时间序列数据集"""
    def __init__(self, sequences, targets):
        self.sequences = torch.FloatTensor(sequences)
        self.targets = torch.FloatTensor(targets)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

def create_sequences(data, target, seq_length=12):
    """创建序列数据"""
    sequences = []
    targets = []
    
    for i in range(len(data) - seq_length):
        seq = data[i:i+seq_length]
        tar = target[i+seq_length]
        sequences.append(seq)
        targets.append(tar)
    
    return np.array(sequences), np.array(targets)

# ============================================================================
# 4. LSTM模型定义
# ============================================================================

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        self.fc1 = nn.Linear(hidden_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]  # 取最后一个时间步
        out = self.dropout(self.relu(self.fc1(out)))
        out = self.dropout(self.relu(self.fc2(out)))
        out = self.fc3(out)
        return out

# ============================================================================
# 5. 训练函数
# ============================================================================

def train_model(model, train_loader, val_loader, epochs=100, lr=0.001, patience=15):
    """训练模型"""
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5 #, verbose=True
    )
    
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # 训练阶段
        model.train()
        train_loss = 0
        for sequences, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs.squeeze(), targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # 验证阶段
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for sequences, targets in val_loader:
                outputs = model(sequences)
                loss = criterion(outputs.squeeze(), targets)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        scheduler.step(val_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
        
        # 早停
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), str(OUTPUT_DIR / 'best_model.pth'))
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
    
    # 加载最佳模型
    model.load_state_dict(torch.load(str(OUTPUT_DIR / 'best_model.pth')))
    
    return model, train_losses, val_losses

# ============================================================================
# 6. 评估指标
# ============================================================================

def calculate_metrics(y_true, y_pred):
    """计算各种评估指标"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    
    # 避免除零错误
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape
    }

# ============================================================================
# 7. 主函数
# ============================================================================

def main():
    print("="*80)
    print("时间序列预测系统")
    print("="*80)
    
    # 加载数据
    print("\n1. 加载数据...")
    train_df = load_data(str(DATA_DIR / 'train.csv'))
    val_df = load_data(str(DATA_DIR / 'val.csv'))
    test_df = load_data(str(DATA_DIR / 'test.csv'))
    
    print(f"训练集大小: {len(train_df)}")
    print(f"验证集大小: {len(val_df)}")
    print(f"测试集大小: {len(test_df)}")
    
    # 合并训练和验证数据用于最终预测
    full_train_df = pd.concat([train_df, val_df], ignore_index=True)
    full_train_df['Week'] = range(len(full_train_df))
    
    # 创建特征
    print("\n2. 创建特征...")
    train_df_feat = create_features(train_df)
    val_df_feat = create_features(val_df)
    full_train_df_feat = create_features(full_train_df)
    
    # 设置序列长度
    seq_length = 12
    
    # 准备训练数据（仅使用Target作为特征，简化版LSTM）
    train_data = train_df['Target'].values.reshape(-1, 1)
    val_data = val_df['Target'].values.reshape(-1, 1)
    
    # 标准化
    scaler = StandardScaler()
    train_data_scaled = scaler.fit_transform(train_data)
    val_data_scaled = scaler.transform(val_data)
    
    # 创建序列
    X_train, y_train = create_sequences(train_data_scaled, train_data_scaled.flatten(), seq_length)
    X_val, y_val = create_sequences(val_data_scaled, val_data_scaled.flatten(), seq_length)
    
    # 创建数据加载器
    train_dataset = TimeSeriesDataset(X_train, y_train)
    val_dataset = TimeSeriesDataset(X_val, y_val)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # 初始化模型
    print("\n3. 初始化LSTM模型...")
    model = LSTMModel(input_size=1, hidden_size=128, num_layers=2, dropout=0.2)
    
    # 训练模型
    print("\n4. 训练模型...")
    model, train_losses, val_losses = train_model(
        model, train_loader, val_loader, 
        epochs=100, lr=0.001, patience=15
    )
    
    # 验证集评估
    print("\n5. 验证集评估...")
    model.eval()
    val_predictions = []
    with torch.no_grad():
        for sequences, _ in val_loader:
            outputs = model(sequences)
            val_predictions.extend(outputs.squeeze().numpy())
    
    val_predictions = np.array(val_predictions).reshape(-1, 1)
    val_predictions_rescaled = scaler.inverse_transform(val_predictions).flatten()
    val_actual = val_df['Target'].values[seq_length:]
    
    val_metrics = calculate_metrics(val_actual, val_predictions_rescaled)
    
    print("\n验证集性能指标:")
    print("-" * 40)
    for metric, value in val_metrics.items():
        print(f"{metric}: {value:.4f}")
    
    # 使用完整训练数据重新训练
    print("\n6. 使用完整训练数据重新训练...")
    full_train_data = full_train_df['Target'].values.reshape(-1, 1)
    full_scaler = StandardScaler()
    full_train_data_scaled = full_scaler.fit_transform(full_train_data)
    
    X_full, y_full = create_sequences(full_train_data_scaled, full_train_data_scaled.flatten(), seq_length)
    full_dataset = TimeSeriesDataset(X_full, y_full)
    full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)
    
    final_model = LSTMModel(input_size=1, hidden_size=128, num_layers=2, dropout=0.2)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(final_model.parameters(), lr=0.001)
    
    # 训练最终模型
    for epoch in range(50):
        final_model.train()
        for sequences, targets in full_loader:
            optimizer.zero_grad()
            outputs = final_model(sequences)
            loss = criterion(outputs.squeeze(), targets)
            loss.backward()
            optimizer.step()
    
    # 测试集预测
    print("\n7. 生成测试集预测...")
    final_model.eval()
    
    # 使用最后seq_length个数据点进行预测
    test_predictions = []
    current_sequence = full_train_data_scaled[-seq_length:].reshape(1, seq_length, 1)
    
    for i in range(len(test_df)):
        with torch.no_grad():
            current_seq_tensor = torch.FloatTensor(current_sequence)
            pred = final_model(current_seq_tensor)
            pred_value = pred.item()
            test_predictions.append(pred_value)
            
            # 更新序列
            current_sequence = np.append(current_sequence[0, 1:, :], [[pred_value]], axis=0).reshape(1, seq_length, 1)
    
    # 反标准化
    test_predictions = np.array(test_predictions).reshape(-1, 1)
    test_predictions_rescaled = full_scaler.inverse_transform(test_predictions).flatten()
    
    # 保存预测结果
    print("\n8. 保存预测结果...")
    submission = pd.DataFrame({
        'Relative Date': test_df.iloc[:, 0],
        'Predict Target': test_predictions_rescaled
    })
    submission.to_csv(str(OUTPUT_DIR / 'submission.csv'), index=False)
    print("预测结果已保存至 submission.csv")
    
    # 可视化
    print("\n9. 生成可视化...")
    plt.figure(figsize=(15, 10))
    
    # 训练和验证损失
    plt.subplot(2, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    
    # 验证集预测 vs 实际
    plt.subplot(2, 2, 2)
    plt.plot(val_actual, label='Actual', marker='o')
    plt.plot(val_predictions_rescaled, label='Predicted', marker='x')
    plt.xlabel('Week')
    plt.ylabel('Target Value')
    plt.title('Validation Set: Actual vs Predicted')
    plt.legend()
    plt.grid(True)
    
    # 完整时间序列
    plt.subplot(2, 1, 2)
    plt.plot(range(len(full_train_df)), full_train_df['Target'], label='Train+Val', marker='o')
    plt.plot(range(len(full_train_df), len(full_train_df) + len(test_df)), 
             test_predictions_rescaled, label='Test Predictions', marker='x', color='red')
    plt.xlabel('Week')
    plt.ylabel('Target Value')
    plt.title('Complete Time Series with Predictions')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'forecast_results.png'), dpi=300, bbox_inches='tight')
    print("可视化结果已保存至 forecast_results.png")
    
    print("\n" + "="*80)
    print("预测完成！")
    print("="*80)

if __name__ == "__main__":
    main()